# Combining & Joining Data

## Loading Libraries

In [ ]:
# Numeric Computing 
import numpy as np

# Data Manipulation
import pandas as pd

# Data Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# PyArrow
import pyarrow as pa

### Adding Rows to Dataframes

In [ ]:
string_pa = pd.ArrowDtype(pa.string())

In [ ]:
df1 = pd.DataFrame({'name': ['John', 'George', 'Ringo'], 
                    'color': ['Blue', 'Blue', 'Purple']})

df2 = pd.DataFrame({'name': ['Paul', 'George', 'Ringo'], 
                    'carcolor': ['Red', 'Blue', np.nan]},
                   index=[3, 1, 2])

In [ ]:
print(df1)

In [ ]:
print(df2)

In [ ]:
pd.concat([df1, df2])

In [ ]:
pd.concat([df1, df2], verify_integrity=True)

In [ ]:
pd.concat([df1, df2], ignore_index=True)

### Adding Columns to Dataframes

In [ ]:
pd.concat([df1, df2], axis=1)

In [ ]:
df1.set_index('name').join(df2.set_index('name')) 

### Joins

In [ ]:
df1.merge(df2)

In [ ]:
print(df1.merge(df2, how='outer'))

In [ ]:
print(df1.merge(df2, how='left'))

In [ ]:
print(df1.merge(df2, how='right'))

In [ ]:
print(df1.merge(df2, how='right', left_on='color', right_on='carcolor'))

In [ ]:
print(df1.merge(df2, how='cross'))

### Joins Indicatiors

In [ ]:
print(df1.merge(df2, how='outer', indicator=True))

### Anti Joins

In [ ]:
print(df1.merge(df2, on='name', how='outer', indicator=True)
.query('_merge != "both"'))

### Merge Validation

In [ ]:
df1.merge(df2, how='right', left_on='color',
    right_on='carcolor', validate='1:m')

### Dirty Devil & Weather Data

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/'\
      'dirtydevil.txt'

df = pd.read_csv(url, skiprows=lambda num: num <34 or num == 35,
                 sep='\t')

In [ ]:
def to_us_mountain_time(df_, time_col, tz_col):
    return (df_
            .assign(**{tz_col: df_[tz_col].replace('MDT',
                       'MST7MDT')})

            .groupby(tz_col)
            [time_col]
            .transform(lambda s: pd.to_datetime(s)
                .dt.tz_localize(s.name, ambiguous=True)
                .dt.tz_convert('US/Mountain'))
           )

In [ ]:
def tweak_river(df_):
    return (df_
      .assign(datetime=to_us_mountain_time(df_, 'datetime', 'tz_cd'))
      .rename(columns={'144166_00060': 'cfs',
                       '144167_00065': 'gage_height'})
    )

In [ ]:
dd = tweak_river(df)
dd

In [ ]:
url = 'https://github.com/mattharrison/datasets/raw/master/data/'\
      'hanksville.csv'

In [ ]:
temp_df = pd.read_csv(url)
def tweak_temp(df_):
    return (df_
           .assign(DATE=pd.to_datetime(df_.DATE)
                .dt.tz_localize('US/Mountain', ambiguous=False))
           .loc[:,['DATE', 'PRCP', 'TMIN', 'TMAX', 'TOBS']]
    )

In [ ]:
temp_df = tweak_temp(temp_df)
temp_df

### Joining Data

In [ ]:
(dd
 .merge(temp_df, left_on='datetime', right_on='DATE')
)

In [ ]:
# (dd
#     .groupby(pd.Grouper(key='datetime', freq='D'))
#     .median()
#     .merge(temp_df, left_index=True, right_on='DATE')
#     )

In [ ]:
(dd
    .groupby(pd.Grouper(key='datetime', freq='D'))
    .median(numeric_only=True)
    .merge(temp_df, left_index=True, right_on='DATE')
    )

In [ ]:
# (dd
#  .groupby(pd.Grouper(key='datetime', freq='D'))
#  .median()
#  .merge(temp_df, left_index=True, right_on='DATE',
#         how='inner', validate='1:1')
# )

In [ ]:
(dd
    .groupby(pd.Grouper(freq='D'))
    .median(numeric_only=True)
    .merge(temp_df, left_index=True, right_on='DATE', how='inner', validate='1:1')
    )

In [ ]:
type(dd.index)

In [ ]:
dd.columns

In [ ]:
dd['datetime'] = pd.to_datetime(dd['datetime'])

In [ ]:
dd = dd.set_index('datetime')

In [ ]:
(dd
    .groupby(pd.Grouper(freq='D'))
    .median(numeric_only=True)
    .merge(temp_df, left_index=True, right_on='DATE', how='inner', validate='1:1')
    )

### Visualization of Merged Data

In [ ]:
# fig, ax = plt.subplots(dpi=600)    

# (dd
#     .groupby(pd.Grouper(key='datetime', freq='D'))
#     .median().merge(temp_df, left_index=True, right_on='DATE', how='inner', validate='1:1')
#     .set_index('DATE')
#     .loc['2014':,['cfs', 'gage_height', 'PRCP', 'TOBS']]
#     .interpolate()
#     .rolling(15)
#     .mean()
#     .plot(subplots=True, figsize=(10,8), ax=ax)
#     )

# fig.suptitle('Dirty Devil Metrics (15 day average)')
# plt.grid(True)
# plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=600)

(dd
    .resample('D')
    .median(numeric_only=True)
    .merge(
        temp_df,
        left_index=True,
        right_on='DATE',
        how='inner',
        validate='1:1'
    )
    .set_index('DATE')
    .loc['2014':, ['cfs', 'gage_height', 'PRCP', 'TOBS']]
    .interpolate()
    .rolling(15)
    .mean()
    .plot(subplots=True, figsize=(10, 8))
)

fig.suptitle('Dirty Devil Metrics (15 day average)')
plt.grid(True)
plt.show()

In [ ]:
fig, ax = plt.subplots(dpi=600)

dd2 = (dd
    .groupby(pd.Grouper(freq='D'))
    .median(numeric_only=True)
    .merge(temp_df, left_index=True, right_on='DATE', how='inner', validate='1:1')
    .query('cfs < 400')
    )

(dd2
    .plot.scatter(x='cfs', y='TOBS', c=dd2.DATE.dt.month, ax=ax, cmap='hsv', alpha=.5)
    )

ax.set_title(
    'Observation Temperature (TOBS) '
    'vs River Flow (cubic feet per sec)\nColored by Month')

plt.grid(True)
plt.show()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fb7807c2-6864-47b0-a018-d85f0d8b2b6c' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>